# pt-simple

Loads MATSim's *pt-simple* scenario with its transit schedule and vehicles, routes the `pt` leg with
RAPTOR, and runs it with transit vehicles, first on the timetable (`vehicles="schedule"`, the default),
then in the queue network (`vehicles="network"`), where the bus waits at stops for its scheduled departures.

In [ ]:
import logging

from mobslim import Controller, ControllerParams, EventRecorder, Scenario, read_network_xml, read_population_xml
from mobslim.events import describe
from mobslim.transit import TransitParams, read_transit

logging.basicConfig(level=logging.INFO, format="%(message)s")

In [ ]:
d = "../scenarios/pt-simple"
network = read_network_xml(f"{d}/network.xml")
population = read_population_xml(f"{d}/population.xml", network)
transit = read_transit(f"{d}/transitSchedule.xml", f"{d}/transitVehicles.xml", network)
s = transit.schedule
s, s.stop_times(0), transit.vehicles.capacity

In [ ]:
recorder = EventRecorder()
result = Controller(
    Scenario(network, population, transit=transit), listeners=[recorder], params=ControllerParams(iterations=3)
).run()
leg = result.scenario.population[0].selected_plan.legs[0]
print(s.describe_rides(leg.route.rides))
[describe(e) for e in recorder.last]

In [ ]:
recorder = EventRecorder()
result = Controller(
    Scenario(network, population, transit=transit.with_params(TransitParams(vehicles="network"))),
    listeners=[recorder], params=ControllerParams(iterations=1),
).run()
[describe(e) for e in recorder.last]